# 03 Baseline: pretrained Whisper-small

Kaggle: Accelerator = GPU, Internet = On, input = `echoscript-clean`.

1. Full FLEURS Hindi test (418 utterances), WER under three normalization settings
2. Our validation clips (618) with a duration-based token cap, to stop runaway repetition

Everything saved to /kaggle/working (download from the Output tab).

In [ ]:
!pip install -q jiwer "datasets<4"


In [ ]:
import os, glob, re, time, math, unicodedata
import numpy as np, pandas as pd, soundfile as sf, torch, jiwer
from datasets import load_dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration

ROOT = os.path.dirname(glob.glob("/kaggle/input/**/manifest.csv", recursive=True)[0])
print("ROOT:", ROOT, "| GPU:", torch.cuda.get_device_name(0))

def eval_norm(t):
    t = unicodedata.normalize("NFC", t).replace("_", " ")
    t = re.sub(r"[,.।॥!?:;\-\"'“”‘’()]", " ", t)
    return re.sub(r"\s+", " ", t).strip().lower()

device = "cuda"
proc = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(device).half().eval()

def decode_batch(arrays, max_new_tokens=None):
    feats = proc.feature_extractor(arrays, sampling_rate=16000,
                                   return_tensors="pt").input_features.to(device).half()
    kw = {} if max_new_tokens is None else {"max_new_tokens": int(min(max_new_tokens, 440))}
    with torch.no_grad():
        ids = model.generate(feats, language="hi", task="transcribe", **kw)
    return proc.batch_decode(ids, skip_special_tokens=True)


## 1. Full FLEURS Hindi test, default decoding (greedy, no token cap)

In [ ]:
fl = load_dataset("google/fleurs", "hi_in", split="test", trust_remote_code=True)
fl = fl.cast_column("audio", Audio(sampling_rate=16000))
print("utterances:", len(fl))

t0 = time.time()
hyp = []
for i in range(0, len(fl), 16):
    batch = fl[i:i+16]["audio"]
    hyp += decode_batch([a["array"] for a in batch])
print("FLEURS decode time (min):", round((time.time() - t0) / 60, 1))

df_fl = pd.DataFrame({"ref": fl["transcription"], "ref_raw": fl["raw_transcription"], "hyp_raw": hyp})
df_fl.to_csv("/kaggle/working/fleurs_baseline_predictions.csv", index=False, encoding="utf-8-sig")

def wer_of(refs, hyps):
    keep = [(r, h) for r, h in zip(refs, hyps) if r.strip()]
    return round(jiwer.wer([r for r, _ in keep], [h for _, h in keep]), 4)

print("A  no normalization, raw_transcription vs raw output :", wer_of(df_fl.ref_raw, df_fl.hyp_raw))
print("B  eval_norm both, transcription                      :",
      wer_of([eval_norm(x) for x in df_fl.ref], [eval_norm(x) for x in df_fl.hyp_raw]))
print("C  eval_norm both, raw_transcription                  :",
      wer_of([eval_norm(x) for x in df_fl.ref_raw], [eval_norm(x) for x in df_fl.hyp_raw]))
print("Josh's table value for the pretrained model: 0.83")


## 2. Our validation clips

Cap on generated tokens = 1.5 x (99th percentile of reference tokens per second) x clip duration + 5.
It is a design choice of mine, not a standard. It only stops runaway repetition; it does not change normal outputs.

In [ ]:
man = pd.read_csv(f"{ROOT}/manifest.csv")
tok = [len(proc.tokenizer(t, add_special_tokens=False).input_ids) for t in man.text]
man["tokens"] = tok
ok = man[man.duration >= 1.0]
tps99 = float(np.percentile(ok.tokens / ok.duration, 99))
print("reference tokens/sec, 50th/99th percentile:",
      round(float(np.percentile(ok.tokens / ok.duration, 50)), 1), round(tps99, 1))

def cap_for(d): return max(10, 1.5 * tps99 * d + 5)

val = man[man.split == "val"].sort_values("duration").reset_index(drop=True)
t0 = time.time()
hyp_v = []
for i in range(0, len(val), 16):
    chunk = val.iloc[i:i+16]
    arrs = [sf.read(f"{ROOT}/clips/{c}")[0] for c in chunk.clip]
    hyp_v += decode_batch(arrs, max_new_tokens=cap_for(chunk.duration.max()))
print("val decode time (min):", round((time.time() - t0) / 60, 1))

val["hyp_raw"] = hyp_v
val["ref_n"] = [eval_norm(x) for x in val.text]
val["hyp_n"] = [eval_norm(x) for x in val.hyp_raw]
val.to_csv("/kaggle/working/val_baseline_predictions.csv", index=False, encoding="utf-8-sig")

print("WER on our 618 val clips (eval_norm, capped):", wer_of(val.ref_n, val.hyp_n))
runaway = (val.hyp_n.str.split().str.len() > 3 * val.ref_n.str.split().str.len() + 5)
print("clips where output has > 3x the reference words + 5:", int(runaway.sum()), "of", len(val))
print(val.loc[runaway, ["clip", "duration", "ref_n", "hyp_n"]].head(5).to_string())
